# Getting started with libstb

Install the requirements first: `pip install -r requirements.txt`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import libstb

libstb.__file__

## Make an image

`libstb.Image(array)` wraps a writable, C-contiguous `uint8` array without copying it.

In [ ]:
y, x = np.mgrid[0:200, 0:320]
r = (x * 255 // 319).astype(np.uint8)
g = (y * 255 // 199).astype(np.uint8)
b = np.full_like(r, 128)
img = libstb.Image(np.dstack([r, g, b]))
img.width, img.height, img.channels

In [ ]:
plt.imshow(img.numpy())
plt.axis('off');

## Encode in memory

`to_png`, `to_jpg`, `to_bmp` and `to_tga` return the file's bytes. Each has its own settings.

In [ ]:
sizes = {
    "png (compression=9)": len(img.to_png(compression=9)),
    "jpg (quality=20)": len(img.to_jpg(quality=20)),
    "jpg (quality=90)": len(img.to_jpg(quality=90)),
    "bmp": len(img.to_bmp()),
}
for name, n in sizes.items():
    print(f"{name:22s} {n:>8,} bytes")

## Decode again

`Image.open` takes a path or bytes. `ImageInfo.read` reads only the header.

In [ ]:
data = img.to_png()
print(libstb.ImageInfo.read(data))
back = libstb.Image.open(data)
print("lossless round trip:", np.array_equal(back.numpy(), img.numpy()))

jpg = libstb.Image.open(img.to_jpg(quality=20))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(back.numpy()); axes[0].set_title("PNG (exact)")
axes[1].imshow(jpg.numpy()); axes[1].set_title("JPEG quality 20")
for a in axes: a.axis("off")

## The functional API

`imread` and `imwrite` work with plain numpy arrays: the format comes from the file extension, and options go to the formats that have them.

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())

libstb.imwrite(tmp / "pattern.png", img)                 # an Image or an array
libstb.imwrite(tmp / "pattern.jpg", img, quality=30)
print(libstb.iminfo(tmp / "pattern.png"))                # header only
pixels = libstb.imread(tmp / "pattern.png")              # just the ndarray
print(pixels.shape, np.array_equal(pixels, img.numpy()))

try:
    libstb.imwrite(tmp / "bad.png", img, quality=30)     # png has no quality
except ValueError as e:
    print(e)

## Channels and flipping

Ask for a channel count or a vertical flip while decoding.

In [ ]:
rgba = libstb.Image.open(data, channels=4)
flipped = libstb.Image.open(data, flip=True)
print("rgba channels:", rgba.channels, "| alpha:", rgba.numpy()[0, 0, 3])
plt.imshow(flipped.numpy()); plt.axis("off");

## Sharing pixels with other frameworks

`Image` speaks DLPack, so numpy, torch, jax and cupy can take the pixels without a copy. The layout is fixed: `uint8`, `(height, width, channels)`, on the CPU.

In [ ]:
shared = np.from_dlpack(img)
print("shares memory:", np.shares_memory(shared, img.numpy()))
back = libstb.Image.from_dlpack(shared)               # and back to an Image
print("from_dlpack shares memory:", np.shares_memory(back.numpy(), shared))
# torch.from_dlpack(img) and jax.numpy.from_dlpack(img) work the same way.
# For another layout, convert on their side:
#     torch.from_dlpack(img).permute(2, 0, 1).float() / 255